# MMA Assignment #3

# <font color=#FF0000>Problem 1: The Markowitz Problem</font>

## Question 1 a
Formulate an NLP to devise the optimal portfolio that minimizes the portfolio risk subject to
non-negative return. What are the optimal solution and value?

In [1]:
import gurobipy as gb
from gurobipy import *
import numpy as np
import pandas as pd

In [2]:
cov_mat = {0:[ 0.001059,  0.001299,  0.000827,  0.001012,  0.000980],
       1 : [0.001299,  0.002794,  0.001224,  0.001597,  0.001700],
       2 : [0.000827,  0.001224,  0.001082,  0.001073,  0.001293],
       3 : [0.001012,  0.001597,  0.001073,  0.002045,  0.001405],
       4 : [0.000980,  0.001700,  0.001293,  0.001405,  0.002147]}

cov_matrix=pd.DataFrame(cov_mat)
avg_rors=[ 0.00069488,  0.00640171,  0.00693084, -0.00107527,  0.00687519]
ilen=5
columns = ['FTSE 100', 'DAX', 'DJIA', 'DJ Asian Titans 50', 'Russell 2000']


In [3]:
prob_1_a=gb.Model("Question 1a")
prob_1_a.Params.NonConvex = 2

Set parameter Username
Academic license - for non-commercial use only - expires 2023-09-13
Set parameter NonConvex to value 2


In [4]:
#df = pd.read_excel (r'C:\Users\raymo\Documents\Education\McGill\McGill Masters\Semseter F22\MGSC 662\Assignments\Assignment #3\Markowitz.xlsx', sheet_name='Markowitz')
#df.columns = df.iloc[1]
#df=df.iloc[2:,]
#will sort by ascending
#df=df.sort_values(by='Date')
#df=df.iloc[:,1:]
#ilen=len(df.columns)
#closing price of stocks
#closing = np.transpose(np.array(df)) # matrix of daily closing prices
#difference from previous day (excludes opeening price)
#difference=np.diff(closing)
#creates the relative type and converts to float for cov matrix
#relative_difference=np.divide(difference,closing[:,:-1])
#relative_difference=relative_difference.astype(float)

In [5]:
#o=np.cov(relative_difference)
# mean price change
#avg_rors= np.mean(relative_difference, axis=1)

In [6]:
########## Create function variables ##########
STK=prob_1_a.addVars(ilen,vtype=GRB.CONTINUOUS,lb=0, ub=1, name = ["Fraction for stock "+i for i in columns])
avg_ror=prob_1_a.addVar(vtype=GRB.CONTINUOUS, name = "Average Return")

In [7]:
portrisk=sum(cov_matrix[i][j]*STK[i]*STK[j] for i in range(ilen) for j in range(ilen))

In [8]:
########## Set objective function ########## 
#minimize is okay because increases will show up as newgative sigmas therefore the minimization will optimally look for lowest sigma
prob_1_a.setObjective(portrisk,GRB.MINIMIZE)


In [9]:
########## Set Constraints ##########
prob_1_a.addConstr(sum(STK[i] for i in range(ilen))==1)

#positivity constraint 
prob_1_a.addConstr(avg_ror>=0)
prob_1_a.addConstr(avg_ror==sum(avg_rors[i]*STK[i] for i in range(ilen)))

<gurobi.Constr *Awaiting Model Update*>

In [10]:
########## Optimize the Problem ##########
prob_1_a.optimize()


Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 3 rows, 6 columns and 12 nonzeros
Model fingerprint: 0x7af4a616
Model has 15 quadratic objective terms
Coefficient statistics:
  Matrix range     [7e-04, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e-03, 7e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 1 columns
Presolve time: 0.02s
Presolved: 2 rows, 5 columns, 10 nonzeros
Presolved model has 15 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 4
 AA' NZ     : 1.500e+01
 Factor NZ  : 2.100e+01
 Factor Ops : 9.100e+01 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   2.47677527e+05 -2.47677527e+05  5.00e+03 1.08e-06  2.98e+05     0s
   1   5.68767036e

In [11]:
for v in prob_1_a.getVars():
    print(v.varname,"=",v.x)

Fraction for stock FTSE 100 = 0.5236061096424093
Fraction for stock DAX = 9.649523261796118e-07
Fraction for stock DJIA = 0.47636351372278285
Fraction for stock DJ Asian Titans 50 = 2.1197310702091368e-05
Fraction for stock Russell 2000 = 8.214371789354823e-06
Average Return = 0.003665482568798189


In [12]:
########## Print Results ##########
p1_adict={}
alist=[]
alistp=[]
alistr=[]
for i in STK:
    alist.append(STK[i].X)
    alistp.append(round(STK[i].X*100,2))
    alistr.append(avg_rors[i]*STK[i].X)
p1_adict["Fraction"]=alist
p1_adict["Percent"]=alistp
p1_adict["Expected Return"]=alistr
p1_adict["Avg Returns"]=avg_rors
df1_a=pd.DataFrame(p1_adict,index=columns)
df1_a.loc["Total"] = df1_a.sum()

#Formating
def df_style(val):
    return "font-weight: bold"
df1_a['Percent'] = df1_a['Percent'].map('{:,.1f}%'.format)

df1_a['Expected Return'] = df1_a['Expected Return'].map('$ {:,.12f}'.format)
last_row = pd.IndexSlice[df1_a.index[df1_a.index == "Total"], :]
df1_a = df1_a.style.applymap(df_style, subset=last_row)

df1_a

,Fraction,Percent,Expected Return,Avg Returns
FTSE 100,0.523606,52.4%,$ 0.000363843413,0.000695
DAX,0.000001,0.0%,$ 0.000000006177,0.006402
DJIA,0.476364,47.6%,$ 0.003301599295,0.006931
DJ Asian Titans 50,0.000021,0.0%,$ -0.000000022793,-0.001075
Russell 2000,0.000008,0.0%,$ 0.000000056475,0.006875
Total,1.000000,100.0%,$ 0.003665482569,0.019827


In [13]:
print("The optimized lowest portfolio risk is {:,}".format(prob_1_a.objVal))

The optimized lowest portfolio risk is 0.0009484859399315851


## Question 1 b
Assume the initial allocation is 20% in each index. Changing the position requires incurring
transaction costs. Formulate an NLP to find the optimal portfolio to minimize risk subject to
non-negative return. What are the optimal solution and value?

In [20]:
import gurobipy as gb
from gurobipy import *
import numpy as np
import pandas as pd

In [21]:
cov_mat = {0:[ 0.001059,  0.001299,  0.000827,  0.001012,  0.000980],
       1 : [0.001299,  0.002794,  0.001224,  0.001597,  0.001700],
       2 : [0.000827,  0.001224,  0.001082,  0.001073,  0.001293],
       3 : [0.001012,  0.001597,  0.001073,  0.002045,  0.001405],
       4 : [0.000980,  0.001700,  0.001293,  0.001405,  0.002147]}

cov_matrix=pd.DataFrame(cov_mat)
avg_rors=[ 0.00069488,  0.00640171,  0.00693084, -0.00107527,  0.00687519]
ilen=5
columns = ['FTSE 100', 'DAX', 'DJIA', 'DJ Asian Titans 50', 'Russell 2000']

In [22]:
prob_1_b=gb.Model("Question 1b")
prob_1_b.Params.NonConvex = 2

Set parameter NonConvex to value 2


In [23]:
#df = pd.read_excel (r'C:\Users\raymo\Documents\Education\McGill\McGill Masters\Semseter F22\MGSC 662\Assignments\Assignment #3\Markowitz.xlsx', sheet_name='Markowitz')
#df.columns = df.iloc[1]
#df.columns = ['Date', 'FTSE 100', 'DAX', 'DJIA', 'DJ Asian Titans 50', 'Russell 2000']
#df=df.iloc[2:,]
#will sort by ascending
#df=df.sort_values(by='Date')
#df=df.iloc[:,1:]
#df
#o=np.cov(relative_difference)
#cov_matrix=pd.DataFrame(o)
# mean price change
#avg_rors= np.mean(relative_difference, axis=1)            
#avg_rors
#closing price of stocks
#closing = np.transpose(np.array(df)) # matrix of daily closing prices
#difference from previous day (excludes opeening price)
#difference=np.diff(closing)
#creates the relative type and converts to float for cov matrix
#relative_difference=np.divide(difference,closing[:,:-1])
#relative_difference=relative_difference.astype(float)

In [24]:
########## Create function variables ##########
STK1_b=prob_1_b.addVars(ilen,vtype=GRB.CONTINUOUS,lb=0, ub=1, name = ["Fraction for stock "+i for i in (columns)])
avg_ror=prob_1_b.addVar(vtype=GRB.CONTINUOUS, name = "Average Return")

In [25]:
portrisk=sum(cov_matrix[i][j]*STK1_b[i]*STK1_b[j] for i in range(ilen) for j in range(ilen))

In [26]:
########## Set objective function ########## 
#minimize is okay because increases will show up as newgative sigmas therefore the minimization will optimally look for lowest sigma
prob_1_b.setObjective(portrisk,GRB.MINIMIZE)


In [27]:
########## Set Constraints ##########
prob_1_b.addConstr(sum(STK1_b[i] for i in range(ilen))==1)

#positivity constraint
prob_1_b.addConstr(avg_ror>=0)
prob_1_b.addConstr(avg_ror==(sum(avg_rors[i]*STK1_b[i] for i in range(ilen))-sum((STK1_b[i]-.2 )**2 for i in range(ilen))))

<gurobi.QConstr Not Yet Added>

In [28]:
########## Optimize the Problem ##########
prob_1_b.optimize()


Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 2 rows, 6 columns and 6 nonzeros
Model fingerprint: 0x000317c5
Model has 15 quadratic objective terms
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [4e-01, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e-03, 7e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [2e-01, 2e-01]
Presolve removed 1 rows and 0 columns

Continuous model is non-convex -- solving as a MIP

Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 17 rows, 11 columns, 41 nonzeros
Presolved model has 15 quadratic objective terms
Presolved model has 5 bilinear constraint(s)
Variable types: 11 continuous, 0 integer (0 binary)

Root relaxation: objective 9.484784e-04, 8 iterations, 0.00 seconds (0.00 w

In [29]:
for v in prob_1_b.getVars():
    print(v.varname,"=",v.x)

Fraction for stock FTSE 100 = 0.2343852814165866
Fraction for stock DAX = 0.16140387885733704
Fraction for stock DJIA = 0.22930504362711976
Fraction for stock DJ Asian Titans 50 = 0.1897965884463585
Fraction for stock Russell 2000 = 0.1851092076525981
Average Return = 0.0


In [30]:
########## Print Results ##########
p1_bdict={}
blist=[]
blistp=[]
blistr=[]
for i in STK1_b:
    blist.append(STK1_b[i].X)
    blistp.append(round(STK1_b[i].X*100,2))
    blistr.append(avg_rors[i]*STK1_b[i].X)
p1_bdict["Fraction"]=blist
p1_bdict["Percent"]=blistp
p1_bdict["Expected Return"]=blistr
df1_b=pd.DataFrame(p1_bdict,index=columns)
df1_b.loc["Total"] = df1_b.sum()

#Formating
def df_style(val):
    return "font-weight: bold"
df1_b['Percent'] = df1_b['Percent'].map('{:,.1f}%'.format)

df1_b['Expected Return'] = df1_b['Expected Return'].map('$ {:,.12f}'.format)
last_row = pd.IndexSlice[df1_b.index[df1_b.index == "Total"], :]
df1_b = df1_b.style.applymap(df_style, subset=last_row)

df1_b

,Fraction,Percent,Expected Return
FTSE 100,0.234385,23.4%,$ 0.000162869644
DAX,0.161404,16.1%,$ 0.001033260825
DJIA,0.229305,22.9%,$ 0.001589276569
DJ Asian Titans 50,0.189797,19.0%,$ -0.000204082578
Russell 2000,0.185109,18.5%,$ 0.001272660973
Total,1.000000,100.0%,$ 0.003853985434


In [31]:
print("The optimized lowest portfolio risk is {:,}".format(prob_1_b.objVal))

The optimized lowest portfolio risk is 0.0012892653894843336


# <font color=#FF0000>Problem 2: The CYCOM Corporation Problem</font>

## Question 2 a
Construct and solve a nonlinear optimization model to determine the number of engineers to assign to each project that will maximize the expected contribution to profit of the six projects minus the cost of assigning engineers to the projects and the start-up costs of the projects. What
is the optimal solution? (Rounded to one decimal place)


In [26]:
import gurobipy as gb
from gurobipy import *
import numpy as np
import pandas as pd

In [27]:
prob_2_a=gb.Model("Question 2a")
prob_2_a.Params.NonConvex = 2
eng=6
projs=6
success_pr_prob=[1.1,.5,2.5,1.6,2.2,2.4]
cost_proj=[325000,200000,490000,125000,710000,240000]
profit=[1750000,700000,1300000,800000,1450000,1300000]
cost_eng=150000

Set parameter NonConvex to value 2


In [28]:
########## Create function variables ##########
x2_a=prob_2_a.addVars(eng,vtype=GRB.CONTINUOUS, lb=0, ub=25, name = ["Fraction of engineer " + str(i) for i in range(eng)])
z2_a=prob_2_a.addVars(eng,vtype=GRB.CONTINUOUS, name = ["dummy denominator " + str(i) for i in range(eng)])
n2_a=prob_2_a.addVars(eng,vtype=GRB.CONTINUOUS, name = ["dummy numerator " + str(i) for i in range(eng)])
p2_a=prob_2_a.addVars(eng,vtype=GRB.CONTINUOUS ,name = ["Probability of success" + str(i) for i in range(eng)])

In [29]:
########## Set objective function ########## 

prob_2_a.setObjective(sum(p2_a[i]*profit[i]-(1-p2_a[i])*cost_proj[i] for i in range(projs))-sum(x2_a[i] * cost_eng for i in range(eng)),GRB.MAXIMIZE)

In [30]:
########## Set Constraints ##########

#engineers max =25
prob_2_a.addConstr(sum(x2_a[i] for i in range(projs))<=25)

#proability of project success defined
for i in range(projs):
    #Setting denominiator
    prob_2_a.addConstr(z2_a[i]==(x2_a[i]+success_pr_prob[i]))
    #setting condition
    prob_2_a.addConstr(n2_a[i]*z2_a[i]==1)
    #setting prability 
    prob_2_a.addConstr(p2_a[i]==x2_a[i]*n2_a[i])

In [31]:
########## Optimize the Problem ##########
prob_2_a.optimize()

for i in prob_2_a.getVars():
        if "Fraction" in i.varName:
            print(i.varName,i.x)

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 7 rows, 24 columns and 18 nonzeros
Model fingerprint: 0x906a91d3
Model has 12 quadratic constraints
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [2e+05, 2e+06]
  Bounds range     [3e+01, 3e+01]
  RHS range        [5e-01, 3e+01]
  QRHS range       [1e+00, 1e+00]

Continuous model is non-convex -- solving as a MIP

Presolve time: 0.00s
Presolved: 55 rows, 24 columns, 126 nonzeros
Presolved model has 12 bilinear constraint(s)
Variable types: 24 continuous, 0 integer (0 binary)

Root relaxation: objective 4.023955e+07, 12 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 4.0240e+07    0

In [32]:
########## Print Results ##########
#print objective coeficients
#creeate dictionary to make dataframe
p2_adict={}
alist=[]
alistp=[]
j=0
for i in x2_a:
    val=x2_a[i].X
    p=val/(val+success_pr_prob[j])
    alist.append(val)
    alistp.append(p*profit[j]-(1-p)*cost_proj[j])
    j+=1
p2_adict["Allocated Engineers"]=alist
p2_adict["Expected profit"]=alistp
df2_a=pd.DataFrame(p2_adict)

#Get sum
df2_a.loc["Engineer Cost"] = [0,-sum(x2_a[i].X for i in range(projs))*cost_eng]
df2_a.loc["Total"] = df2_a.sum()

#Formating
def df_style(val):
    return "font-weight: bold"

last_row = pd.IndexSlice[df2_a.index[df2_a.index == "Total"], :]
df2_a['Expected profit'] = df2_a['Expected profit'].map('${:,.2f}'.format)
df2_a = df2_a.style.applymap(df_style, subset=last_row)


df2_a

,Allocated Engineers,Expected profit
0,2.800018,"$1,164,746.31"
1,1.231209,"$440,066.09"
2,2.963395,"$480,912.22"
3,1.541172,"$328,838.24"
4,3.425370,"$605,255.64"
5,2.565972,"$555,734.89"
Engineer Cost,0.000000,"$-2,179,070.55"
Total,14.527137,"$1,396,482.85"


In [33]:
print("The optimized expected profit is therefore ${:,}".format(round(prob_2_a.objVal,1)))

The optimized expected profit is therefore $1,396,484.5


## Question 2 b
Construct and solve an optimization model that minimizes the standard deviation of the
contribution to profit subject to the constraint that the expected contribution to profit (minus the
cost of engineers) is at least $1.1 million. What is the optimal solution? (Rounded to one decimal
place)

In [34]:
import gurobipy as gb
from gurobipy import *
import numpy as np
import pandas as pd

In [35]:
prob_2_b=gb.Model("Question 2a")
prob_2_b.Params.NonConvex = 2


eng=6
projs=6
success_pr_prob=[1.1,.5,2.5,1.6,2.2,2.4]
cost_proj=[-325, -200, -490, -125, -710, -240]
profit=[1750, 700,1300, 800, 1450, 1300]
cost_eng=150
min_prof=1100


Set parameter NonConvex to value 2


In [36]:
pvar2_b=[0]*6
for i in range(projs):
    pvar2_b[i]=(profit[i]+cost_proj[i])**2

In [37]:
########## Create function variables ##########
x2_b=prob_2_b.addVars(eng,vtype=GRB.CONTINUOUS, lb=0, ub=25, name = ["Fraction of engineer " + str(i) for i in range(eng)])
p2_b=prob_2_b.addVars(eng,vtype=GRB.CONTINUOUS, lb=0, ub=1, name = ["Probability of success" + str(i) for i in range(eng)])


In [38]:
########## Set objective function ########## 
prob_2_b.setObjective(sum(((profit[i]-cost_proj[i])**2)*p2_b[i]*(1-p2_b[i]) for i in range(projs)),GRB.MINIMIZE)

In [39]:
########## Set Constraints ##########
#total number of engineers available
prob_2_b.addConstr(sum(x2_b[i] for i in range(projs)) <= 25)

#proability of project success defined enables the fraction of x/(x+#)
for i in range(projs):
    prob_2_b.addConstr( x2_b[i] == p2_b[i]*(x2_b[i]+success_pr_prob[i]))

#sum of all projects must be equal to at least 1.1 million
prob_2_b.addConstr(sum(p2_b[i]*profit[i] + (1-p2_b[i])*cost_proj[i] - x2_b[i]*cost_eng for i in range(eng))>=min_prof)


<gurobi.Constr *Awaiting Model Update*>

In [40]:
########## Optimize the Problem ##########
prob_2_b.optimize()

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 2 rows, 12 columns and 18 nonzeros
Model fingerprint: 0x7e1c3ff8
Model has 6 quadratic objective terms
Model has 6 quadratic constraints
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [5e-01, 3e+00]
  Objective range  [8e+05, 5e+06]
  QObjective range [2e+06, 9e+06]
  Bounds range     [1e+00, 3e+01]
  RHS range        [3e+01, 3e+03]

Continuous model is non-convex -- solving as a MIP

Presolve time: 0.00s
Presolved: 33 rows, 19 columns, 73 nonzeros
Presolved model has 12 bilinear constraint(s)
Variable types: 19 continuous, 0 integer (0 binary)

Root relaxation: objective 2.336596e+05, 19 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Nod

In [41]:
for i in prob_2_b.getVars():
        if "Fraction" in i.varName:
            print(i.varName,i.x)

Fraction of engineer 0 5.051188581982796
Fraction of engineer 1 1.6467656216754636
Fraction of engineer 2 4.2081685329174165
Fraction of engineer 3 1.4739860340122364
Fraction of engineer 4 5.983077201321211
Fraction of engineer 5 3.046289721024364


In [42]:
########## Print Results ##########
#print objective coeficients
#creeate dictionary to make dataframe
import math
p2_bdict={}
blist=[]
blistp=[]
blists=[]
j=0
for i in x2_b:
    val=x2_b[i].X
    p=val/(val+success_pr_prob[j])
    blist.append(val)
    blistp.append(p*profit[j]-(1-p)*cost_proj[j])
    blists.append(math.sqrt(p*(1-p)*(profit[j]-cost_proj[j])**2))
    j+=1
p2_bdict["Allocated Engineers"]=blist
p2_bdict["Expected profit"]=blistp
p2_bdict["Standard Dev"]=blists
df2_b=pd.DataFrame(p2_bdict)
#Print data frame

#Get sum
df2_b.loc["Engineer Cost"] = [0,float(-1*sum(x2_b[i].X for i in range(projs))*cost_eng),0]
df2_b.loc["Total"] = df2_b.sum()

#Formating
def df_style(val):
    return "font-weight: bold"

last_row = pd.IndexSlice[df2_b.index[df2_b.index == "Total"], :]
df2_b['Expected profit'] = df2_b['Expected profit'].map('${:,.2f}'.format)
df2_b['Standard Dev'] = df2_b['Standard Dev'].map('${:,.2f}'.format)
df2_b = df2_b.style.applymap(df_style, subset=last_row)


df2_b


,Allocated Engineers,Expected profit,Standard Dev
0,5.051189,"$1,495.17",$795.16
1,1.646766,$583.55,$380.42
2,4.208169,$998.13,$865.50
3,1.473986,$448.66,$462.11
4,5.983077,"$1,251.05",$957.66
5,3.046290,$832.89,$764.56
Engineer Cost,0.000000,"$-3,211.42",$0.00
Total,21.409476,"$2,398.04","$4,225.40"


In [43]:
print("The minimized standard deviation is therefore ${:,}".format(round(math.sqrt(prob_2_b.objVal),0)))

The minimized standard deviation is therefore $1,800.0
